# Convert a list of fits files into a WWT ingestible WTML file

We will use `wwt_data_formats` to built a `Folder` containing a single imageset per image. This is the most bandwidth consuming option, but is also easy. But it also requires lying about the header. For now, since there is so little, we will just reproject


We have to do a couple extra steps to avoid a couple bugs
- Even though the tiler does not care what the projection is (it get's reprojected), the WTML builder looks at the original header and will break if it sees something that is not a TAN projected. Since these are tiny, SIN = TAN. But if near a pole, this assumption will break
- We need to set the blank value. by default it puts nan in empty pixels, but the WWT shader does not reliably capture nan. 

_Both of these issues should get fixed upstream to avoid this extra processing_

**You should be able to just run the cells in order to have this complete**

In [ ]:
import glob
from pathlib import Path
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp
from wwt_data_formats.folder import Folder
from wwt_data_formats.imageset import ImageSet
from wwt_data_formats.place import Place
from wwt_data_formats.enums import Bandpass, ProjectionType, Classification


def sin_to_tan(src_path: Path, out_path: Path):
    """
        Reproject SIN to TAN"
    """
    with fits.open(src_path) as hdul:
        data = hdul[0].data
        source_wcs = WCS(hdul[0].header).celestial
        header = hdul[0].header

    # Build a target TAN with identical CRVAL/CRPIX/CDELT and shape
    
    new_header = source_wcs.to_header()
    new_header["CTYPE1"] = "RA---TAN"
    new_header["CTYPE2"] = "DEC--TAN"
    new_wcs = WCS(new_header)
    out, _ = reproject_interp((data, source_wcs), new_wcs, shape_out=data.shape)
    out = out.astype(np.float32)
    
    out_hdr = new_wcs.to_header()
    out_hdr["NAXIS"]  = 2
    out_hdr["NAXIS1"] = data.shape[1]
    out_hdr["NAXIS2"] = data.shape[0]
    out_hdr['DATAMIN'] = float(np.nanmin(out))
    out_hdr['DATAMAX'] = float(np.nanmax(out))
    
    # fill in the rest of the header from the original, excluding WCS keywords
    for key, value in header.items():
        if key not in out_hdr and not key.startswith(('CR', 'CTYPE', 'CDELT', 'CRVAL', 'CUNIT')):
            out_hdr[key] = value

    fits.writeto(out_path, out.astype(np.float32), out_hdr, overwrite=True)
    return out_hdr, data.shape[1], data.shape[0], data

In [ ]:
# https://wwt-data-formats.readthedocs.io/en/latest/api/wwt_data_formats.imageset.ImageSet.html#wwt_data_formats.imageset.ImageSet
fits_files = Path('./source').glob("*.pbcor.fits")
REPROJ =  Path("reprojected")
REPROJ.mkdir(parents=True, exist_ok=True)

folder = Folder()
folder.name = "ALMAGAL Images"
folder.children = []

pix_cut_low = -np.inf
pix_cut_high = np.inf
for src in fits_files:
    src = Path(src)
    dst = REPROJ / src.name
    # if dst.exists():
    #     with fits.open(dst) as hdul:
    #         hdr = hdul[0].header
    #         h, w = hdul[0].data.shape
    # else:
    hdr, w, h, data = sin_to_tan(src, dst)
    
    place = Place()

    imgset = ImageSet()
    imgset.name = src.stem
    imgset.url = f"./reprojected/{src.name}"
    imgset.file_type = "fits"
    imgset.tile_levels = 0
    imgset.base_tile_level = 0
    imgset.band_pass = Bandpass.MICROWAVE
    imgset.projection = ProjectionType.SKY_IMAGE
    imgset.bottoms_up = True
    imgset.set_position_from_wcs(hdr, width=w, height=h, place = place)
    imgset.data_max = float(np.nanmax(data))
    imgset.data_min = float(np.nanmin(data))
    plow, phigh = np.nanpercentile(data, [0.5, 99.5])
    pix_cut_low = max(pix_cut_low, plow)
    pix_cut_high = min(pix_cut_high, phigh)
    imgset.pixel_cut_low = plow
    imgset.pixel_cut_high = phigh
    
    place.foreground_image_set = imgset
    place.name = src.stem
    place.classification = Classification.DARK_NEBULA
    
    place.data_set_type = imgset.data_set_type


    folder.children.append(place)
    




In [ ]:
# uncomment to use the same default pixel cut for all images
# for place in folder.children:
#     place.foreground_image_set.pixel_cut_low = float(pix_cut_low)
#     place.foreground_image_set.pixel_cut_high = float(pix_cut_high)

In [ ]:
with open("index.wtml", "w") as f:
    folder.write_xml(f)